# CS781 Phase 6: Abstractive Summarization of Indian Supreme Court Judgments
## Lead ML Research Engineering & Championship Audit Notebook

**Hardware Platform:** NVIDIA GeForce RTX 4060 Laptop GPU (8,188 MiB VRAM), Intel Core i7-13700H, Windows 11  
**Software Stack:** Python 3.11, PyTorch 2.5.1+cu121, CUDA 12.1, Transformers 4.47.1, HuggingFace Datasets  
**Primary Metric:** Validation `BERTScore F1` (`roberta-large`, layer 17, unscaled, CPU offload)  
**Constraint Mandate:** Strictly student-trained Phase 1 checkpoints (V1, V2, V3, V4); zero external LLMs, generator weights, or hosted APIs.


### 1. Environment Verification & Hardware Diagnostics
Asserts CUDA device availability, GPU memory capacity, PyTorch version, and directory setup.


In [ ]:
import sys
import os
import torch
import transformers
import bert_score

print(f"Python Executable: {sys.executable}")
print(f"PyTorch Version:   {torch.__version__} (CUDA Available: {torch.cuda.is_available()})")
if torch.cuda.is_available():
    print(f"Device Name:       {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:        {torch.cuda.get_device_properties(0).total_memory / (1024**2):.0f} MiB")
print(f"Transformers:      {transformers.__version__}")
print(f"BERTScore Version: {bert_score.__version__}")


### 2. Dataset Reconciliation & Group-Aware Deterministic Splitting
Filters 2 blank reference summaries (`['299', '4799']`), identifies 27 duplicate document groups, and applies SHA-256 group-aware 90/10 deterministic splitting (seed 42) with zero document overlap.


In [ ]:
from src.dataset import build_deterministic_splits, load_test_dataset
from src.tokenizer import load_phase1_tokenizer

train_recs, val_recs = build_deterministic_splits()
test_recs = load_test_dataset()
tokenizer = load_phase1_tokenizer()

print(f"Train Split: {len(train_recs):,} document-summary pairs")
print(f"Val Split:   {len(val_recs):,} document-summary pairs")
print(f"Test Set:    {len(test_recs):,} evaluation judgments")
print(f"Vocabulary:  {tokenizer.vocab_size:,} byte-level BPE tokens")


### 3. Metric Calibration: RoBERTa-Large Layer 17 Sensitivity & Truncation
Demonstrates metric behavior:
- Reference summaries average ~1,002 tokens; RoBERTa's 512-token limit truncates 75.06% of references.
- RoBERTa-Large Layer 17 detects operative disposition reversals with statistical significance.


In [ ]:
from src.evaluator import LegalSummarizationEvaluator

evaluator = LegalSummarizationEvaluator(device="cpu", batch_size=4)

cand_allowed = "The appeal is allowed and the judgment of the High Court is set aside."
cand_dismissed = "The appeal is dismissed and the judgment of the High Court is confirmed."
ref_gold = "The appeal filed by the appellant is allowed. The conviction is set aside."

score_match = evaluator.evaluate([cand_allowed], [ref_gold])
score_mismatch = evaluator.evaluate([cand_dismissed], [ref_gold])

print(f"Disposition Allowed Match -> BERTScore F1: {score_match['bertscore_f1']:.4f}")
print(f"Disposition Dismiss Mismatch -> BERTScore F1: {score_mismatch['bertscore_f1']:.4f}")
print(f"Delta Sensitivity: {score_match['bertscore_f1'] - score_mismatch['bertscore_f1']:+.4f}")


### 4. Source Packing Strategy Study
Controlled comparison on held-out validation set across Front-Only, Lead-Tail (67/33), and Lead-Middle-Tail (40/30/30).


In [ ]:
import pandas as pd
import os

packing_csv = os.path.join("..", "results", "packing_comparison.csv")
if os.path.exists(packing_csv):
    df_packing = pd.read_csv(packing_csv)
    display(df_packing)
else:
    print("Packing comparison CSV not found.")


### 5. Decoding Optimization Grid & Repetition Penalty
Shows the empirical discovery that increasing length budget to 250 tokens and greedy repetition penalty to 1.15 increases BERTScore F1 to **0.8323** while eliminating loop degeneracies.


In [ ]:
decoding_csv = os.path.join("..", "results", "decoding_sweep.csv")
if os.path.exists(decoding_csv):
    df_decoding = pd.read_csv(decoding_csv)
    display(df_decoding)


### 6. Official Multi-Model Championship Leaderboard
Evaluates all required experiment families (E1–E4) plus V1-Wide capacity growth under the unified benchmarking protocol.


In [ ]:
lead_csv = os.path.join("..", "results", "official_championship_leaderboard.csv")
if os.path.exists(lead_csv):
    df_lead = pd.read_csv(lead_csv)
    display(df_lead)


### 7. Publication Figures Showcase
Loads and visualizes all 10 championship figures generated from disk.


In [ ]:
from IPython.display import Image, display

figure_names = [
    "training_loss_curves.png",
    "validation_bertscore_by_step.png",
    "final_bertscore_comparison.png",
    "bertscore_precision_recall.png",
    "rouge_comparison.png",
    "generation_length_distribution.png",
    "decoding_sweep_results.png",
    "v1_capacity_and_adapter_comparison.png",
    "resource_efficiency.png",
    "per_example_score_differences.png",
]

for fname in figure_names:
    fpath = os.path.join("..", "figures", fname)
    if os.path.exists(fpath):
        print(f"=== {fname} ===")
        display(Image(filename=fpath, width=700))


### 8. Official Submission Schema & Integrity Audit
Verifies that all required submission JSON files contain exactly 100 test items, valid string summaries, and matching test IDs.


In [ ]:
import json
import glob

sub_files = glob.glob(os.path.join("..", "submissions", "*.json"))
print(f"Found {len(sub_files)} submission files:")

test_recs = load_test_dataset()
expected_ids = set(str(r["id"]) for r in test_recs)

for sf in sub_files:
    fname = os.path.basename(sf)
    with open(sf, "r", encoding="utf-8") as f:
        data = json.load(f)
    assert len(data) == 100, f"{fname} must have 100 items, got {len(data)}"
    sub_ids = set(str(d["id"]) for d in data)
    assert sub_ids == expected_ids, f"{fname} has ID mismatch!"
    non_empty = all(len(d.get("summary", "").strip()) > 0 for d in data)
    assert non_empty, f"{fname} has blank summaries!"
    avg_len = sum(len(d["summary"].split()) for d in data) / len(data)
    print(f"  [PASS] {fname:35s} | 100/100 IDs Verified | Avg Words: {avg_len:.1f}")
